## Code review

In [0]:
'''def load_events(spark, path):
    df = spark.read.csv(path)
    df = df.dropDuplicates()
    df = df.filter(df.spend != None)
    for row in df.collect():
        if row['spend'] < 0:
            df = df.filter(df.event_id != row['event_id'])
    return df.cache()'''

"def load_events(spark, path):\n    df = spark.read.csv(path)\n    df = df.dropDuplicates()\n    df = df.filter(df.spend != None)\n    for row in df.collect():\n        if row['spend'] < 0:\n            df = df.filter(df.event_id != row['event_id'])\n    return df.cache()"

| Code                                             | What's wrong?                                        | Why it matters at scale?                                                                        |
| ------------------------------------------------ | ---------------------------------------------------- | ----------------------------------------------------------------------------------------------- |
| `spark.read.csv(path)`                           | No schema is provided. Spark infers data types.      | Schema inference is slow and can produce incorrect data types.                                  |
| `dropDuplicates()`                               | Removes duplicates using all columns.                | Expensive operation; if only `event_id` defines uniqueness, use `dropDuplicates(["event_id"])`. |
| `df.filter(df.spend != None)`                    | Incorrect way to check NULLs.                        | Spark may not handle this correctly. Use `df.filter(df.spend.isNotNull())`.                     |
| `df.collect()`                                   | Brings the entire dataset to the driver.             | Causes memory issues or crashes with large datasets.                                            |
| `for row in df.collect():`                       | Loops through rows in Python instead of using Spark. | Loses Spark's parallel processing and becomes very slow.                                        |
| `if row['spend'] < 0:`                           | Row-by-row filtering.                                | Spark is designed for distributed operations, not Python loops.                                 |
| `df = df.filter(df.event_id != row['event_id'])` | Re-runs a DataFrame filter for every bad row.        | Multiple full scans of the dataset make performance terrible.                                   |
| `return df.cache()`                              | Caches data without checking if it will be reused.   | Wastes cluster memory if the DataFrame is used only once.                                       |
